# Predicting Air Turbulence Using Machine Learning
By: Chee Qian Wen
<br>Feb 25
## Part 2 of 5: Data Cleaning and Merging
In Part 1, the focus was on importing and extracting the necessary data. Please refer to Part 1 to see how the following data were imported:
1) Pilot reports (PIREPs): 'PIREPs_all.csv'
2) Elevation data: 'elevation.csv'
3) Aircraft specifications: 'ICAO_aircraft_type.csv'
4) Meteoreological data 'weather_all.csv'

Part 2 involves cleaning the data and concatenating all the data into a single dataset in preparation for analyses.

### 2.1. Import Libraries
First, the necessary libraries required are imported.

In [3]:
import pandas as pd
import numpy as np
import pytz
from timezonefinder import TimezoneFinder
from scipy.spatial import cKDTree

### 2.2. Cleaning PIREPs Data

In [5]:
# import csv file as dataframe
df = pd.read_csv('PIREPs_all.csv', low_memory = False)

#### 2.2.1. 'LAT' feature
This feature contains the latitude coordinate for each report. 

In this section, latitude is converted to a number. The latitudinal bounds for US mainland are also specified and data outside of US mainland bounds are excluded.

In [7]:
# convert LAT data to numeric
df['LAT'] = pd.to_numeric(df['LAT'], errors='coerce')

In [8]:
# exclude rows with missing coordinates
df = df[df['LAT'].notna()]

In [9]:
# define lower and upper latitudinal bounds for US mainland
lower_lat = 24.396308
upper_lat = 49.384358

# exclude data outside bounds
df = df[(df['LAT'] >= lower_lat) & (df['LAT'] <= upper_lat)]

#### 2.2.2. 'LON' feature
This feature contains the longitude coordinate for each report.

In this section, longitude is converted to number. The longitudinal bounds for US mainland are also specified and data outside of US mainland bounds are excluded.

In [11]:
# convert LAT data to numeric
df['LON'] = pd.to_numeric(df['LON'], errors='coerce')

In [12]:
# exclude rows with missing coordinates
df = df[df['LON'].notna()]

In [13]:
# define lower and upper longitudinal bounds for US mainland
lower_lon = -125.000000
upper_lon = -66.934570

# exclude data outside bounds
df = df[(df['LON'] >= lower_lon) & (df['LON'] <= upper_lon)]

#### 2.2.3. 'DATETIME' Feature
This feature contains data for the date and time of each pilot report.
In this section, the date and time data are converted to datetime format. The original data is in UTC and needs to be converted into US time for accuracy.

In [15]:
# rename datetime feature
df.rename(columns={'VALID': 'DATETIME'}, inplace=True)

In [16]:
# extract only the first 12 digits of the data and convert to datetime format
df['DATETIME'] = df['DATETIME'].astype(str)
df['DATETIME'] = pd.to_datetime(df['DATETIME'].str[:12], format='%Y%m%d%H%M', errors='coerce')

In [17]:
# drop rows which have missing datetime data
df = df[df['DATETIME'].notna()]

In [18]:
# initialize the TimezoneFinder
tf = TimezoneFinder()

# Ensure 'DATETIME' is timezone-aware (set it to UTC)
df['DATETIME'] = pd.to_datetime(df['DATETIME']).dt.tz_localize('UTC')

# Function to convert UTC time to local time
def convert_to_local_time(row):
    # Get the timezone from lat/lon
    time_zone_str = tf.timezone_at(lng=row['LON'], lat=row['LAT'])
    
    if time_zone_str:
        local_tz = pytz.timezone(time_zone_str)
        return row['DATETIME'].astimezone(local_tz)  # Convert UTC to local time
    else:
        return row['DATETIME']  # If no timezone found, keep UTC

# Convert 'DATETIME' to local time
df.loc[:, 'DATETIME'] = pd.to_datetime(df.apply(lambda row: convert_to_local_time(row).replace(tzinfo=None), axis=1))

C:\Users\qianw\AppData\Local\Temp\ipykernel_26476\691786050.py:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '<DatetimeArray>
['2014-12-31 16:00:00', '2014-12-31 16:01:00', '2014-12-31 16:14:00',
 '2014-12-31 17:15:00', '2014-12-31 16:15:00', '2014-12-31 19:17:00',
 '2014-12-31 16:22:00', '2014-12-31 18:25:00', '2014-12-31 18:25:00',
 '2014-12-31 18:26:00',
 ...
 '2024-11-03 00:15:00', '2024-11-03 00:35:00', '2024-11-03 00:50:00',
 '2024-11-03 01:55:00', '2024-11-03 00:07:00', '2024-11-03 01:22:00',
 '2024-11-03 01:30:00', '2024-11-03 02:50:00', '2024-11-03 02:28:00',
 '2024-11-03 01:35:00']
Length: 2325356, dtype: datetime64[ns]' has dtype incompatible with datetime64[ns, UTC], please explicitly cast to a compatible dtype first.
  df.loc[:, 'DATETIME'] = pd.to_datetime(df.apply(lambda row: convert_to_local_time(row).replace(tzinfo=None), axis=1))


#### 2.2.4. 'URGENT' feature
This feature contains data for whether the report was submitted as an urgent report ('T') or not ('F'). In this section, the data is converted to a True / False boolean data. Since most reports are not urgent, all missing values are imputed at False.

In [20]:
# map 'T' to True and 'F' to False, and all missing data as False
df.loc[:, 'URGENT'] = df['URGENT'].map({'T': True}).astype('boolean').fillna(False).astype(bool)

#### 2.2.5. 'TURBULENCE' feature
This feature contains raw reports of turbulence. The description of turbulence varies between pilots, as there is no standardized format or phraseology for reporting turbulence. For example, moderate turbulence can be written as '/TB MOD', '/TB MODERATE', '/TB MDR', etc. 

To clean this data, each unique report is scanned for turbulence levels and their various forms of writing, in line with Turbli's method of interpreting PIREPs (https://turbli.com/blog/new-us-turbulence-map-based-on-1-million-pilot-reports/).
* Light turbulence: 'LIGHT','LGT','LHT','LIT','LT','LGHT','LHGT'
* Moderate turbulence: 'MODERATE','MOD','MD','MID','MDR'
* Severe turbulence: 'SEVERE','SVR','SEVR','SVER','HEAVY','HVY'
* Extreme turbulence: 'EXTREME','EXTRM','XTRM'
* Choppiness: 'CHOP'

In this section, a new feature `TURB_CLEANED` is created to reflect the cleaned turbulence data, based on a CSV file. This new feature reflects only the turbulence category (with or without CHOP) and removes all other irrelevant data.

In [22]:
# import csv file as dataframe
turb_cleaned = pd.read_csv('turbulence_cleaning.csv', low_memory = False)

# left join on TURBULENCE data
df = pd.merge(df, turb_cleaned, on ='TURBULENCE', how ='left') 
df = df[df['TURB_CLEANED'].notna()]

#### 2.2.6. 'FL' feature
This feature contains data on the flight level. Flight level is reported in feet.

In this section, flight level is converted to a number. Furthermore, since aircrafts are prone to turbulence when ascending or descending, it is common to only analyse turbulence above 10,000ft. Data below 10,000ft are thus excluded. Outlier data above 50,000 ft are also excluded.

In [24]:
# convert FL data to numeric
df['FL'] = pd.to_numeric(df['FL'], errors='coerce')

In [25]:
# exclude rows with missing FL data and records below 10,000ft
df = df.dropna(subset=['FL'])
df = df[(df['FL'] >= 10000) & (df['FL'] <= 50000)]

#### 2.2.7. 'AIRCRAFT' feature
This feature contains the aircraft type for each report. This feature is quite messy but difficult to clean, as some pilots reported flight numbers (e.g., CA170) instead of aircraft types. This feature was cleaned only for aircraft types that were obvious (e.g., '737' as Boeing 737 'B737'), following the ICAO list of aircraft types (https://www.icao.int/publications/DOC8643/Pages/Search.aspx).

In this section, a new feature `AIRCRAFT_TYPE` is created to reflect the cleaned aircraft data, based on a CSV file.

In [27]:
# import csv file as dataframe
aircraft_cleaned = pd.read_csv('aircraft_cleaned.csv', low_memory = False)

# left join on AIRCRAFT data
df = pd.merge(df, aircraft_cleaned, on ='AIRCRAFT', how ='left') 

### 2.3. Joining Elevation Data
The elevation data ('elevation.csv') is then merged to the PIREPs dataset based on longitudes and latitudes.

Since elevation data is in meters, data is converted to feet to align with flight levels.

In [29]:
# import csv file as dataframe
elevation = pd.read_csv('elevation.csv', low_memory = False)

# since lon and lat have been rounded to 1 d.p. in the elevation data, create temporary columns to round lon and lat in the dataset
df['lon_rounded'] = df['LON'].round(1)
df['lat_rounded'] = df['LAT'].round(1)

In [30]:
# left join on rounded LON and LAT data
df = pd.merge(df, elevation, left_on=['lon_rounded', 'lat_rounded'], right_on=['LON', 'LAT'], how='left')

# drop the rounded LON and LAT data and the duplicate LON and LAT data
df = df.drop(columns=['lon_rounded', 'lat_rounded', 'LON_y', 'LAT_y'])

# rename LON and LAT columns
df.rename(columns={'LON_x': 'LON', 'LAT_x': 'LAT'}, inplace=True)

# exclude rows with missing elevation data
df = df[df['Elevation'].notna()]

In [31]:
# convert elevation data to feet
df['Elevation'] *= 3.28084

### 2.4. Joining Aircraft Specs Data
The aircraft specifications data ('ICAO_aircraft_type.csv') is then merged to the PIREPs and elevation dataset based on aircraft type.

In [33]:
# import csv file as dataframe
aircraft_specs = pd.read_csv('ICAO_aircraft_type.csv', low_memory = False)

# since the data has duplicates, we will drop duplicates
aircraft_specs = aircraft_specs.drop_duplicates(subset='AIRCRAFT_TYPE', keep='first')

# left join on AIRCRAFT_TYPE data
df = pd.merge(df, aircraft_specs, on = 'AIRCRAFT_TYPE', how = 'left')

# exclude rows with missing aircraft specs data
df = df[df['MANUFACTURER'].notna()]

### 2.5. Joining Metereological Data
The metereological data is then merged to the PIREPs, elevation, and aircraft specs dataset based on nearest station coordinates and date.

In [35]:
weather_df = pd.read_csv('weather_all.csv', low_memory = False)

In [36]:
weather_df['day'] = pd.to_datetime(weather_df['day'])
df['DATE'] = pd.to_datetime(df['DATETIME']).dt.normalize()

In [37]:
# Create a unique list of weather stations and build a KDTree
stations = weather_df[['st_lon', 'st_lat']].drop_duplicates().to_numpy()
tree = cKDTree(stations)

processed_rows = 0

# Function to find the nearest station
def find_nearest_station(row):
    global processed_rows
    dist, idx = tree.query([row['LON'], row['LAT']])
    processed_rows += 1
    print(f"Processed rows: {processed_rows}", end='\r')  # Prints on the same line
    return pd.Series(stations[idx])  # Return nearest station coordinates

# Apply function to find nearest stations for each row in df
df[['nearest_lon', 'nearest_lat']] = df.apply(find_nearest_station, axis=1)

# Merge on date and nearest station coordinates
merged_df = df.merge(
    weather_df,
    left_on=['DATE', 'nearest_lon', 'nearest_lat'],
    right_on=['day', 'st_lon', 'st_lat'],
    how='left'
)

# Drop extra columns
merged_df.drop(columns=['nearest_lon','nearest_lat','DATE','precip_in','day','st_elev'], inplace=True)

print("\nProcessing complete!")

Processed rows: 1670276
Processing complete!


In [38]:
# drop rows with no weather data
df = merged_df.dropna(subset=['max_temp_f', 'min_temp_f', 'avg_wind_speed_kts', 'avg_wind_drct', 'avg_rh'])

In [39]:
# convert temperatures to deg C
df[['max_temp_f', 'min_temp_f']] = (df[['max_temp_f', 'min_temp_f']].copy() - 32) * 5.0/9.0

# Rename the columns
df.rename(columns={'max_temp_f': 'max_temp_c', 'min_temp_f': 'min_temp_c'}, inplace=True)

C:\Users\qianw\AppData\Local\Temp\ipykernel_26476\422694632.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[['max_temp_f', 'min_temp_f']] = (df[['max_temp_f', 'min_temp_f']].copy() - 32) * 5.0/9.0
C:\Users\qianw\AppData\Local\Temp\ipykernel_26476\422694632.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={'max_temp_f': 'max_temp_c', 'min_temp_f': 'min_temp_c'}, inplace=True)


### 2.6. Export Combined Data to CSV

In [68]:
df.to_csv('combined_data.csv')

## Proceed to Part 3.

There is now one combined dataset, 'combined_data.csv'.

Part 3 explains how new features are engineered based on this combined data.